# AI MCQ Pipeline — Gemma-4-E4B-it (Unsloth)
This notebook integrates the modular `src/` pipeline.


### 1. Setup Environment


### 2. Inject Pipeline Modules


In [12]:
!mkdir -p src

In [13]:
%%writefile src/config.py
"""
Centralized configuration for Hackathon MCQ Pipeline.
Optimized for Maximum Inference Speed.
"""

# ─── Model Settings ───
PRIMARY_MODEL = "google/gemma-4-E4B-it"
FALLBACK_MODEL = "Qwen/Qwen2.5-7B-Instruct"

MAX_SEQ_LENGTH = 1024

MAX_NEW_TOKENS = 32
TEMPERATURE = 0.0
BATCH_SIZE = 64

# ─── vLLM Settings (Docker mode) ───
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL_NAME = "gemma4"
GPU_MEMORY_UTILIZATION = 0.95

# ─── Data Paths (SỬA LẠI CHO CHUẨN COLAB) ───
DATA_DIR = "data"          # Hoặc dùng tuyệt đối: "/content/data"
OUTPUT_DIR = "output"      # Hoặc dùng tuyệt đối: "/content/output"
OUTPUT_FILENAME = "pred.csv"

# ─── Inference Settings ───
RETRY_COUNT = 1
SAVE_INTERVAL = 100

# VÔ HIỆU HÓA SELF-CONSISTENCY (Tăng tốc x3 lần)
SELF_CONSISTENCY_N = 1
SELF_CONSISTENCY_TEMP = 0.0

Overwriting src/config.py


In [14]:
%%writefile src/classifier.py
"""
Question Classifier — Extremely Fast Rule-based classification.
Priority: RC > MATH > SAFETY > KNOWLEDGE

Optimizations:
1. RegEx Union (|) & Pre-compilation: Evaluates all patterns in a single C-level pass.
2. Context Expansion: Scans multiple RC keywords.
3. Deep Scan: Scans choices text for MATH indicators.
"""
import re

# ─── 1. COMPILE PATTERNS ONCE (SPEED OPTIMIZATION) ───

# Mở rộng các keyword thường thấy trong câu hỏi Đọc Hiểu
RC_PATTERN = re.compile(
    r"(đoạn thông tin|đoạn trích|đoạn văn|theo bài đọc|dựa vào văn bản|dựa trên thông tin)",
    re.IGNORECASE
)

# Gộp toàn bộ Safety keywords thành 1 chuỗi Regex duy nhất
SAFETY_PATTERN = re.compile(
    r"(làm thế nào để.*trái|cách.*phá hoại|làm giả|xúc phạm.*biểu tượng|xúc phạm.*lãnh tụ|"
    r"lợi dụng.*tham nhũng|phát tán.*mật|làm suy yếu|chống lại đường lối|kích động.*chia rẽ|"
    r"phá hoại.*tinh thần|tránh.*bị phát hiện|lợi dụng quyền lực|phá hoại.*đoàn kết|"
    r"tránh.*cung cấp thông tin.*cơ quan|phi pháp|phi đạo đức|trái pháp luật|bất hợp pháp|"
    r"hack|lừa đảo|rửa tiền|chế tạo bom|vũ khí|tự tử|chất độc)",
    re.IGNORECASE
)

# Gộp toàn bộ Math/Science keywords thành 1 chuỗi Regex duy nhất
MATH_PATTERN = re.compile(
    r"(\$|\\frac|\\sqrt|\\int|\\sum|\\lim|\\log|\\sin|\\cos|\\tan|\\text|"
    r"tính\b|giải\b|phương trình|bất phương trình|hệ phương trình|biểu thức|đạo hàm|"
    r"tích phân|giới hạn|cực trị|nghiệm|ma trận|định thức|ước\b|bội\b|số nguyên tố|"
    r"ƯCLN|BCNN|hoán vị|tổ hợp|chỉnh hợp|xác suất|thống kê|vận tốc|gia tốc|lực\b|"
    r"điện trở|cường độ|hiệu điện thế|công suất|năng lượng|tần số|bước sóng|"
    r"quang phổ|phóng xạ|chu kỳ|động năng|thế năng|điện dung|từ thông|mol\b|"
    r"nguyên tử khối|phân tử khối|nồng độ|thể tích|khối lượng|dung dịch|pH|"
    r"bằng bao nhiêu|kết quả là|giá trị.*bằng|tìm\s+(?:x|y|z|n|m|a|b|c)|tìm giá trị)",
    re.IGNORECASE
)

# ─── 2. CORE CLASSIFICATION LOGIC ───

def classify_question(question: dict) -> str:
    """
    Classify a question into one of 4 categories with O(1) compiled regex.
    """
    text = question.get("question", "")

    if not text:
        return "KNOWLEDGE"

    # 1. Reading Comprehension (Highest Priority)
    if RC_PATTERN.search(text):
        return "READING_COMPREHENSION"

    # 2. Math/Science
    # Mẹo: Gộp cả nội dung câu hỏi và các đáp án lại để tìm dấu hiệu Toán học (VD: LaTeX)
    choices_dict = question.get("choices", {})
    if isinstance(choices_dict, list):
        choices_text = " ".join(str(c) for c in choices_dict)
    elif isinstance(choices_dict, dict):
        choices_text = " ".join(str(v) for v in choices_dict.values())
    else:
        choices_text = ""

    combined_text = f"{text} {choices_text}"

    if MATH_PATTERN.search(combined_text):
        return "MATH"

    # 3. Safety/Refusal
    if SAFETY_PATTERN.search(text):
        return "SAFETY"

    # 4. General Knowledge (Default)
    return "KNOWLEDGE"


def classify_all(questions: list) -> dict:
    """
    Classify a list of questions and return runtime statistics.
    """
    stats = {"READING_COMPREHENSION": 0, "MATH": 0, "SAFETY": 0, "KNOWLEDGE": 0}

    for q in questions:
        q_type = classify_question(q)
        q["_type"] = q_type  # Annotate in-place for pipeline routing
        stats[q_type] += 1

    return stats

Overwriting src/classifier.py


In [15]:
%%writefile src/prompts.py
"""
Prompt Templates for Vietnamese MCQ answering.
Optimized for MAXIMUM SPEED (Direct Answering, No Chain-of-Thought)
and HIGH ACCURACY (Strict formatting, Recency Bias optimization).
"""

import re

LETTER_MAP = {i: chr(65 + i) for i in range(26)}  # 0->A, 1->B, ..., 25->Z

def format_choices(choices: list) -> str:
    """Format choices dynamically: A. xxx, B. xxx, etc."""
    return "\n".join(f"{LETTER_MAP[i]}. {c}" for i, c in enumerate(choices))

# ─── System Prompts ───

# Khóa chặt hành vi của model: Không giải thích, không lặp lại câu hỏi.
SYSTEM_PROMPT = (
    "Bạn là một hệ thống chấm điểm trắc nghiệm AI siêu tốc và chính xác. "
    "YÊU CẦU TỐI THƯỢNG: Trả lời trực tiếp, KHÔNG giải thích, KHÔNG phân tích, KHÔNG sinh thêm bất kỳ văn bản nào khác. "
    "Chỉ xuất ra đúng một dòng duy nhất theo định dạng: **Đáp án: X** "
    "(Trong đó X là ký tự của đáp án đúng nhất: A, B, C, D...)."
)

# ─── Prompt Builders ───

def _split_passage_and_question(text: str) -> tuple:
    """Split RC question text into (passage, question_only)."""
    parts = re.split(r'\nCâu hỏi\s*:', text, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), "Câu hỏi: " + parts[1].strip()
    return text, ""


def build_rc_prompt(question: dict) -> list:
    """
    Reading Comprehension: Tránh 'Lost in the Middle' bằng cách
    đặt Câu hỏi và Lựa chọn ở CUỐI CÙNG, ngay sát con trỏ sinh text.
    """
    text = question["question"]
    choices_str = format_choices(question["choices"])
    passage, question_only = _split_passage_and_question(text)

    # Đưa passage lên đầu, đưa câu hỏi xuống cuối để tận dụng Recency Bias của LLM
    user_content = (
        f"<passage>\n{passage}\n</passage>\n\n"
        f"Dựa CHỈ vào đoạn thông tin trên, hãy trả lời câu hỏi sau:\n"
        f"{question_only}\n\n"
        f"Các lựa chọn:\n{choices_str}\n\n"
        f"Không giải thích. Hãy in ra kết quả cuối cùng:\n**Đáp án:"
    )

    return [{"role": "user", "content": user_content}]


def build_math_prompt(question: dict) -> list:
    """
    Math/Science: Bỏ Chain-of-Thought (tốn token). Ép model nhẩm trong
    Hidden States và phun thẳng kết quả để đạt tốc độ O(1) Token.
    """
    text = question["question"]
    choices_str = format_choices(question["choices"])

    user_content = (
        f"Giải bài toán sau. Xác định công thức và tính toán nhẩm để tìm kết quả đúng nhất.\n\n"
        f"Câu hỏi: {text}\n\n"
        f"Các lựa chọn:\n{choices_str}\n\n"
        f"Không giải thích. Hãy in ra kết quả cuối cùng:\n**Đáp án:"
    )

    return [{"role": "user", "content": user_content}]


def build_safety_prompt(question: dict) -> list:
    """
    Safety: Kích hoạt bộ lọc an toàn của model bằng từ khóa "Pháp luật & Đạo đức".
    """
    text = question["question"]
    choices_str = format_choices(question["choices"])

    user_content = (
        f"Cảnh báo: Câu hỏi sau có yếu tố nhạy cảm. "
        f"Hãy chọn đáp án KHÔNG vi phạm pháp luật và phù hợp với đạo đức xã hội. "
        f"Ưu tiên đáp án mang tính chất từ chối hành vi xấu.\n\n"
        f"Câu hỏi: {text}\n\n"
        f"Các lựa chọn:\n{choices_str}\n\n"
        f"Không giải thích. Hãy in ra kết quả cuối cùng:\n**Đáp án:"
    )

    return [{"role": "user", "content": user_content}]


def build_knowledge_prompt(question: dict) -> list:
    """General Knowledge: Trực diện."""
    text = question["question"]
    choices_str = format_choices(question["choices"])

    user_content = (
        f"Trả lời câu hỏi trắc nghiệm kiến thức chung sau:\n\n"
        f"Câu hỏi: {text}\n\n"
        f"Các lựa chọn:\n{choices_str}\n\n"
        f"Không giải thích. Hãy in ra kết quả cuối cùng:\n**Đáp án:"
    )

    return [{"role": "user", "content": user_content}]


# ─── Dispatcher ───

PROMPT_BUILDERS = {
    "READING_COMPREHENSION": build_rc_prompt,
    "MATH": build_math_prompt,
    "SAFETY": build_safety_prompt,
    "KNOWLEDGE": build_knowledge_prompt,
}

def build_prompt(question: dict, q_type: str) -> list:
    """Trả về list các messages theo chuẩn ChatML cho vLLM/Transformers."""
    builder = PROMPT_BUILDERS.get(q_type, build_knowledge_prompt)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(builder(question))
    return messages

def build_prompt_text(question: dict, q_type: str) -> str:
    """
    Dành cho các pipeline sử dụng text thô thay vì Chat Template.
    Tốt nhất là dùng tokenizer.apply_chat_template(build_prompt(...)) ở pipeline ngoài.
    """
    builder = PROMPT_BUILDERS.get(q_type, build_knowledge_prompt)
    msg = builder(question)[0]["content"]

    return f"{SYSTEM_PROMPT}\n\n{msg}"

Overwriting src/prompts.py


In [16]:
%%writefile src/parser.py
"""
Answer parser — Extremely fast extraction of letter answers (A-K).
Optimized for short, direct responses from the LLM.
"""
import re

# ─── PRE-COMPILE PATTERNS (O(1) initialization) ───

# Pattern 1: Ưu tiên bắt chính xác format đã ép trong Prompt
# Bắt các dạng: "Đáp án: A", "**Đáp án: B**", "Chọn C", "Answer: D"
PRIMARY_PATTERN = re.compile(r'(?:đáp án|chọn|là|answer)[\s:>*]*([A-K])\b', re.IGNORECASE)

# Pattern 2: Fallback lấy chữ cái in hoa đứng một mình (A, B, C, D...)
FALLBACK_PATTERN = re.compile(r'\b([A-K])\b')

def parse_answer(text: str, num_choices: int = 4) -> str:
    """
    Extract the answer letter from LLM generated text.

    Args:
        text: Raw LLM output text (expected to be very short, < 32 tokens)
        num_choices: Number of choices to validate the letter (e.g., 4 -> A, B, C, D)

    Returns:
        A single valid letter or "A" as ultimate fallback.
    """
    if not text:
        return "A"

    # Xác định giới hạn đáp án hợp lệ (Ví dụ: 4 choices -> 'D')
    max_valid_char = chr(64 + num_choices)

    def is_valid(char: str) -> bool:
        return 'A' <= char <= max_valid_char

    # ─── BƯỚC 1: Bắt chuẩn Format (Nhanh & Chính xác nhất) ───
    match = PRIMARY_PATTERN.search(text)
    if match:
        ans = match.group(1).upper()
        if is_valid(ans):
            return ans

    # ─── BƯỚC 2: Fallback tìm chữ cái đứng riêng lẻ ───
    # Vì chúng ta giới hạn max_new_tokens=32, text rất ngắn, search sẽ chớp nhoáng
    matches = FALLBACK_PATTERN.findall(text)
    if matches:
        # Lấy chữ cái hợp lệ xuất hiện cuối cùng trong câu (thường là chốt đáp án)
        for ans in reversed(matches):
            ans_upper = ans.upper()
            if is_valid(ans_upper):
                return ans_upper

    # ─── BƯỚC 3: Cứu cánh cuối cùng (Ultimate Fallback) ───
    # Nếu mô hình sinh ra một thứ hoàn toàn không có A, B, C, D
    # Mặc định trả về A để tránh lỗi pipeline, vớt vát 25% xác suất đúng.
    return "A"

Overwriting src/parser.py


In [17]:
%%writefile src/loader.py
"""
Data loader — Robust and Fast Data Ingestion.
Handles public_test.json and potential private_test.csv.
Protects against Excel BOM characters and malformed rows.
"""
import os
import json
import csv

# Sử dụng utf-8-sig để tự động loại bỏ ký tự BOM (\ufeff) nếu file CSV được xuất từ Excel
ENCODING = 'utf-8-sig'

def find_test_file(data_dir: str) -> tuple:
    """
    Auto-detect test file in data directory.
    Priority: CSV > JSON.
    Returns: (file_path, file_type)
    """
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Thư mục dữ liệu không tồn tại: {data_dir}")

    files = [f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))]

    # Ưu tiên 1: File có chứa từ khóa
    for f in files:
        f_lower = f.lower()
        if f_lower.endswith('.csv') and any(kw in f_lower for kw in ['test', 'public', 'private']):
            return os.path.join(data_dir, f), 'csv'

    for f in files:
        f_lower = f.lower()
        if f_lower.endswith('.json') and any(kw in f_lower for kw in ['test', 'public', 'private']):
            return os.path.join(data_dir, f), 'json'

    # Ưu tiên 2: Bất kỳ file nào đúng định dạng
    for f in files:
        if f.endswith('.csv'):
            return os.path.join(data_dir, f), 'csv'
    for f in files:
        if f.endswith('.json'):
            return os.path.join(data_dir, f), 'json'

    raise FileNotFoundError(f"Không tìm thấy file .csv hoặc .json nào trong {data_dir}")


def load_questions_json(file_path: str) -> list:
    """Load questions from JSON format with validation."""
    with open(file_path, 'r', encoding=ENCODING) as f:
        data = json.load(f)

    if not isinstance(data, list) or len(data) == 0:
        raise ValueError("Lỗi format JSON: Cấu trúc phải là một List các câu hỏi.")

    required_keys = {'qid', 'question', 'choices'}
    sample = data[0]
    if not required_keys.issubset(sample.keys()):
        missing = required_keys - set(sample.keys())
        raise ValueError(f"Dữ liệu thiếu các cột bắt buộc: {missing}")

    return data


def load_questions_csv(file_path: str) -> list:
    """
    Load questions from CSV format.
    Optimized to compute schema only once from headers instead of per-row.
    """
    questions = []

    with open(file_path, 'r', encoding=ENCODING) as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

        # Khảo sát schema (cấu trúc cột) một lần duy nhất
        has_choices_col = 'choices' in fieldnames

        # Tìm các cột được đặt tên là A, B, C, D...
        choice_cols = [col for col in fieldnames if len(col) == 1 and col.upper() in "ABCDEFGHIJK"]
        # Đảm bảo thứ tự cột chuẩn
        choice_cols.sort()

        for row in reader:
            qid = str(row.get('qid', '')).strip()
            question_text = str(row.get('question', '')).strip()

            if not qid or not question_text:
                continue

            choices = []

            # Layout 1: Cột 'choices' chứa JSON array (VD: '["Đáp án 1", "Đáp án 2"]')
            if has_choices_col and row.get('choices'):
                try:
                    parsed = json.loads(row['choices'])
                    if isinstance(parsed, list):
                        choices = [str(c).strip() for c in parsed]
                except Exception:
                    pass  # Fallback xuống Layout 2 nếu parse JSON lỗi

            # Layout 2: Lấy dữ liệu từ các cột rời A, B, C, D...
            if not choices and choice_cols:
                choices = [str(row.get(col, '')).strip() for col in choice_cols if str(row.get(col, '')).strip()]

            questions.append({
                "qid": qid,
                "question": question_text,
                "choices": choices
            })

    return questions


def load_questions(file_path: str, file_type: str) -> list:
    """Entry point to load questions."""
    print(f"Đang tải dữ liệu từ: {file_path} (Format: {file_type.upper()})")
    if file_type == 'json':
        return load_questions_json(file_path)
    elif file_type == 'csv':
        return load_questions_csv(file_path)
    else:
        raise ValueError(f"Không hỗ trợ định dạng file: {file_type}")

Overwriting src/loader.py


In [18]:
%%writefile src/output.py
"""
Output utilities — Lightning fast, Crash-proof CSV writing.
Optimized for continuous intermediate saving during Hackathon inference.
"""
import os
import csv

def write_predictions(results: list, output_path: str, is_intermediate: bool = False):
    """
    Write predictions to CSV file using Atomic Write.
    Guarantees file integrity even if Colab/Docker crashes mid-write.

    Args:
        results: list of dicts (from pipeline) or (qid, answer_letter) tuples.
        output_path: path to output CSV file (e.g., /output/pred.csv)
        is_intermediate: boolean to change print verbosity.
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # ─── BƯỚC 1: Chuẩn hóa dữ liệu ───
    # Hỗ trợ nhận vào cả list(tuple) hoặc list(dict) từ pipeline chính
    formatted_results = []
    for item in results:
        if isinstance(item, dict):
            formatted_results.append([item.get("qid", ""), item.get("answer", "A")])
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            formatted_results.append([item[0], item[1]])

    # ─── BƯỚC 2: ATOMIC WRITE (Ghi Nguyên Tử) ───
    # Thay vì ghi trực tiếp vào pred.csv, ta ghi vào một file .tmp
    temp_path = f"{output_path}.tmp"

    with open(temp_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["qid", "answer"])

        # writerows() chạy ở tầng C của Python, nhanh hơn rất nhiều so với vòng lặp writerow()
        writer.writerows(formatted_results)

    # Đổi tên file .tmp thành file thật.
    # Thao tác os.replace là "nguyên tử" trên hệ điều hành: Nó thay thế file trong 1 tích tắc.
    # Nếu hệ thống sập trước lúc này, file pred.csv cũ vẫn an toàn 100%.
    os.replace(temp_path, output_path)

    # ─── BƯỚC 3: Log thông báo ───
    if not is_intermediate:
        print(f"✅ Hoàn tất! Đã lưu thành công {len(formatted_results)} dự đoán vào {output_path}")
    else:
        print(f"💾 [Checkpoint] Đã tự động sao lưu {len(formatted_results)} câu...")

Overwriting src/output.py


In [28]:
%%writefile src/main.py
"""
Hackathon MCQ Pipeline - Main Execution Engine.
Optimized for Google Colab (Unsloth/Transformers).
Includes Auto-Download feature for Colab environments.
"""
import os
import time
from tqdm import tqdm
from unsloth import FastLanguageModel

# Import các module vệ tinh siêu tốc
import config
import loader
import classifier
import prompts
import parser
import output

def main():
    print("🚀 Bắt đầu khởi động Pipeline Hackathon...")
    start_time = time.time()

    # 1. TẢI DỮ LIỆU
    data_path, file_type = loader.find_test_file(config.DATA_DIR)
    questions = loader.load_questions(data_path, file_type)
    print(f"📦 Đã tải {len(questions)} câu hỏi.")

    # 2. PHÂN LOẠI CÂU HỎI (Siêu tốc O(1))
    stats = classifier.classify_all(questions)
    print(f"📊 Thống kê phân loại: {stats}")

    # 3. KHỞI TẠO MÔ HÌNH (Unsloth)
    print(f"🧠 Đang nạp mô hình: {config.PRIMARY_MODEL}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.PRIMARY_MODEL,
        max_seq_length=config.MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    # 4. CHẠY INFERENCE (Vòng lặp vắt kiệt GPU)
    results = []

    gen_kwargs = {
        "max_new_tokens": config.MAX_NEW_TOKENS,
        "temperature": config.TEMPERATURE,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
    }

    print("⚡ Bắt đầu giải đề...")
    for i in tqdm(range(len(questions)), desc="Đang xử lý"):
        q = questions[i]

        messages = prompts.build_prompt(q, q["_type"])

        system_text = ""
        user_text = ""
        for msg in messages:
            if msg["role"] == "system":
                system_text = msg["content"]
            elif msg["role"] == "user":
                user_text = msg["content"]

        text_prompt = (
            f"<start_of_turn>user\n"
            f"{system_text}\n\n{user_text}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )

        # ─── BẢN VÁ LỖI NẰM Ở ĐÂY ───
        # Dùng `text=text_prompt` thay vì `[text_prompt]` để tránh lỗi của Unsloth_Zoo
        inputs = tokenizer(text=text_prompt, return_tensors="pt").to("cuda")
        # ───────────────────────────

        outputs = model.generate(**inputs, **gen_kwargs)

        input_length = inputs.input_ids.shape[1]
        generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        num_choices = len(q.get("choices", []))
        final_answer = parser.parse_answer(generated_text, num_choices=max(4, num_choices))

        results.append((q["qid"], final_answer))

        if (i + 1) % config.SAVE_INTERVAL == 0:
            output.write_predictions(
                results,
                os.path.join(config.OUTPUT_DIR, config.OUTPUT_FILENAME),
                is_intermediate=True
            )

    # 5. LƯU KẾT QUẢ CUỐI CÙNG
    final_output_path = os.path.join(config.OUTPUT_DIR, config.OUTPUT_FILENAME)
    output.write_predictions(results, final_output_path, is_intermediate=False)

    elapsed = time.time() - start_time
    print(f"🏁 Hoàn thành toàn bộ! Thời gian chạy: {elapsed:.2f} giây.")
    print(f"Tốc độ trung bình: {elapsed / len(questions):.2f} giây/câu.")

    # 6. TỰ ĐỘNG TẢI FILE VỀ MÁY (Dành riêng cho Colab)
    try:
        from google.colab import files
        print(f"📥 Đang thử tự động tải file {config.OUTPUT_FILENAME} về máy tính...")
        files.download(final_output_path)
    except Exception as e:
        # Bắt luôn cả lỗi AttributeError khi chạy bằng !python
        print(f"ℹ️ Không thể tự động kích hoạt tải xuống từ Terminal (!python).")
        print(f"✅ Đừng lo! File của bạn đã được lưu an toàn tại: {final_output_path}")
        print("👉 Hãy nhấp chuột phải vào file ở thanh bên trái (Thư mục 📁) để tải về nhé!")

if __name__ == "__main__":
    main()

Overwriting src/main.py


In [26]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes pandas tqdm requests

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lrt32zyp/unsloth_023ee40bf13141d291bb247c52c179c5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lrt32zyp/unsloth_023ee40bf13141d291bb247c52c179c5
  Resolved https://github.com/unslothai/unsloth.git to commit 58c2ec1ebdd4564b4ceda8e602e55f001f590808
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

In [27]:
!python src/main.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🚀 Bắt đầu khởi động Pipeline Hackathon...
Đang tải dữ liệu từ: data/public-test_1780368312.json (Format: JSON)
📦 Đã tải 463 câu hỏi.
📊 Thống kê phân loại: {'READING_COMPREHENSION': 100, 'MATH': 308, 'SAFETY': 1, 'KNOWLEDGE': 54}
🧠 Đang nạp mô hình: google/gemma-4-E4B-it...
==((====))==  Unsloth 2026.6.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 2130/2130 [00:47<00:00, 45.28it/s]
⚡ Bắt đầu giải đề...
Đang xử lý:   0% 0/463 [00:00<?, ?it/s]/usr/local/lib/pytho